# Environmental Impact Analysis with Gemma 4 12B

This revision replaces the legacy Unsloth/Gemma 3n flow with the latest official Gemma 4 multimodal model and fixes the original inference path so image inputs are actually consumed and generated text is returned.

## What changed
- Uses `google/gemma-4-12B-it`, the latest official Gemma 4 multimodal release as of this revision.
- Uses the official `transformers` multimodal processor/model API.
- Captures generated text and parses a structured JSON result.
- Uses a local reference image asset instead of a missing notebook-only attachment path.
- Removes the hardcoded CUDA-only helper bug from the original notebook structure.

In [ ]:
%pip install -q --upgrade torch torchvision "transformers>=5.5.0" accelerate sentencepiece protobuf pillow

In [ ]:
from __future__ import annotations

import json
import re
import textwrap
from pathlib import Path
from typing import Any

import torch
from IPython.display import display
from PIL import Image
from transformers import AutoModelForMultimodalLM, AutoProcessor

MODEL_ID = "google/gemma-4-12B-it"
IMAGE_PATH = Path("assets/alfred_palmer_smokestacks.jpg")
MAX_NEW_TOKENS = 300

if not torch.cuda.is_available():
    print("Warning: CUDA is not available. Gemma 4 12B is designed for GPU execution and may be slow or fail on CPU.")


In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID, padding_side="left")
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
)
model.eval()
print(f"Loaded {MODEL_ID}")

In [ ]:
def load_image(path: Path) -> Image.Image:
    if not path.exists():
        raise FileNotFoundError(f"Missing image asset: {path}")
    return Image.open(path).convert("RGB")

def extract_json_block(text: str) -> dict[str, Any] | None:
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    candidate = cleaned[start : end + 1]
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        return None

def build_messages(prompt: str, image_path: Path | None = None) -> list[dict[str, Any]]:
    content: list[dict[str, Any]] = []
    if image_path is not None:
        content.append({"type": "image", "url": str(image_path)})
    content.append({"type": "text", "text": prompt})
    return [{"role": "user", "content": content}]

def generate(prompt: str, image_path: Path | None = None, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    if image_path is None:
        messages = build_messages(prompt, image_path=None)
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            add_generation_prompt=True,
        ).to(model.device)
    else:
        messages = build_messages(prompt, image_path=image_path)
        try:
            inputs = processor.apply_chat_template(
                messages,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                add_generation_prompt=True,
            ).to(model.device)
        except Exception:
            fallback_messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": prompt},
                    ],
                }
            ]
            chat_text = processor.apply_chat_template(
                fallback_messages,
                tokenize=False,
                add_generation_prompt=True,
            )
            inputs = processor(
                images=load_image(image_path),
                text=chat_text,
                return_tensors="pt",
            ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    return processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

In [ ]:
reference_image = load_image(IMAGE_PATH)
display(reference_image)
print(f"Reference image loaded from: {IMAGE_PATH}")

In [ ]:
smoke_test = generate(
    "In two sentences, explain why multimodal evidence is useful in environmental impact assessment.",
    max_new_tokens=80,
)
print(smoke_test)

In [ ]:
EIA_PROMPT = textwrap.dedent("""
You are a senior environmental impact analyst.

Inspect the image and return ONLY valid JSON with these keys:
- hazard_level: one of ["low", "medium", "high", "critical"]
- summary: one sentence
- visible_evidence: 3 to 5 short strings
- likely_impact_factors: 3 to 5 short strings
- likely_processes: 2 to 4 short strings
- recommendations: 3 to 5 short strings
- uncertainty: 1 to 3 short strings
- confidence: a number from 0 to 1

Rules:
- Do not use markdown or code fences.
- Do not invent details that are not clearly supported by the image.
- If the image is ambiguous, say so in uncertainty.
""").strip()

raw_analysis = generate(EIA_PROMPT, image_path=IMAGE_PATH, max_new_tokens=MAX_NEW_TOKENS)
print(raw_analysis)

In [ ]:
analysis = extract_json_block(raw_analysis)

if analysis is None:
    print("Could not parse JSON. Inspect raw_analysis above.")
else:
    print(json.dumps(analysis, indent=2, ensure_ascii=False))
    output_path = Path("gemma4_eia_analysis.json")
    output_path.write_text(json.dumps(analysis, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Saved structured result to {output_path}")

## Optional tweaks
- If you need a lighter model, try `google/gemma-4-E4B-it`.
- If you have more GPU headroom, try `google/gemma-4-26B-A4B-it` or `google/gemma-4-31B-it`.
- Increase `MAX_NEW_TOKENS` if you want longer rationales.